In [1]:
import win32com.client as com
import os
import pandas as pd
import geopandas as gpd
from shapely import wkt

In [2]:
folder = r'C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara'
visum_network = os.path.join(folder, "Red Base GDL", "RedBase 120826", "RedBase 300726 - conectores_final - finalfilt.ver")

Visum = com.Dispatch("Visum.Visum")
Visum.LoadVersion(visum_network)
C = com.constants

In [3]:
connectors = pd.DataFrame({
    "ZoneNo": [i[1] for i in Visum.Net.Connectors.GetMultiAttValues("ZoneNo")],
    "NodeNo": [i[1] for i in Visum.Net.Connectors.GetMultiAttValues("NodeNo")],
    "Length": [i[1] for i in Visum.Net.Connectors.GetMultiAttValues("Length")],
    "CreatedBy": [i[1] for i in Visum.Net.Connectors.GetMultiAttValues("CREATED_BY")],
    "highway": [i[1] for i in Visum.Net.Connectors.GetMultiAttValues(r"Node\Distinct:InLinks\HIGHWAY")],
    "clave_ageb": [i[1] for i in Visum.Net.Connectors.GetMultiAttValues(r"Zone\CLAVE_AGEB")],
    "geometry": [i[1] for i in Visum.Net.Connectors.GetMultiAttValues("WKTPolyWGS84")],
})

connectors["geometry"] = connectors["geometry"].apply(wkt.loads)
connectors = gpd.GeoDataFrame(connectors, geometry="geometry", crs="EPSG:4326")

In [4]:
connectors = connectors[['clave_ageb', 'NodeNo', 'Length', 'highway', 'CreatedBy', 'geometry']]
connectors = connectors.rename(columns={'NodeNo':'node_id', 'Length':'length', 'CreatedBy':'created_by'})
connectors['length'] = connectors['length']*1000
connectors = connectors.drop_duplicates(subset=['clave_ageb'], keep='first')

In [5]:
connectors

,clave_ageb,node_id,length,highway,created_by,geometry
0,1403900010026,133628.0,496.287583,"primary,primary_link,residential",algorithm,"LINESTRING (-103.3838 20.7084, -103.3883 20.7098)"
10,1403900010030,134632.0,814.929797,"primary,secondary",algorithm,"LINESTRING (-103.3722 20.7057, -103.3644 20.7053)"
20,140390001005A,139513.0,126.890444,"primary,residential",algorithm,"LINESTRING (-103.3661 20.708, -103.3666 20.7069)"
32,1403900010064,136453.0,170.565731,"primary_link,residential",algorithm,"LINESTRING (-103.3589 20.7073, -103.3605 20.7075)"
42,1403900010083,135667.0,362.380395,residential,algorithm,"LINESTRING (-103.3524 20.7125, -103.3505 20.7097)"
...,...,...,...,...,...,...
19294,141240103,3285.0,6016.253136,"service,unclassified",algorithm,"LINESTRING (-102.96 20.5491, -103.016 20.5358)"
19300,1412401760160,3260.0,966.703007,tertiary,algorithm,"LINESTRING (-103.0848 20.5251, -103.0886 20.5331)"
19310,1412401760599,8939.0,158.082748,residential,algorithm,"LINESTRING (-103.0876 20.5181, -103.0864 20.5191)"
19318,1412401760601,9432.0,84.711997,"residential,unclassified",algorithm,"LINESTRING (-103.0786 20.5255, -103.0794 20.5255)"


In [7]:
visum_objects = "Visum Objects"
connectors.to_file(os.path.join(folder, visum_objects, "Connectors", "connectors.shp"))